This is following - 113  

Part A: We will divide our documents into CHUNKS

In [1]:
import os
import glob 
import tiktoken
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from sklearn.manifold import TSNE
import plotly.graph_objects as go
 

/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Langchain related Imports 

from langchain_chroma import Chroma # the newer, recommended approach
# from langchain_community.vectorstores import Chroma - older approach


from langchain_openai import OpenAIEmbeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_16932/3474719954.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [3]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


Let's read in all  data into a dictionary

In [4]:
# How many files in all the documents?

knowledge_base_path = "knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

Found 15 files in the knowledge base


Just check the size of knowldege by getting token count 

In [5]:
# How many characters in all the documents?

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Total characters in knowledge base: 18,195


In [6]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode( entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 3,400


We could think like why cant we pass these whole token as single context to gpt models as the latest gpt models can handle these many tokens easily . 

But we need to understand if the document is very huge , then sending the whole context for a simple question which is present across few tokens is waste of resources . 

In [7]:
# Load in everything in the knowledgebase using LangChain's loaders

knowledge_base_path = "knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)

#folders = glob.glob("knowledge_base/*")
documents = []
for file in files:
    doc_type = os.path.basename(file)
    #print(f"folder :{file} , doc_cat : {doc_type}")
    #loader = DirectoryLoader(file, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    loader = TextLoader(file)
    loader_docs = loader.load()
    for doc in loader_docs:
        #doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")
 

Loaded 15 documents


In [12]:
documents


[Document(metadata={'source': 'knowledge_base/Shari_Professional.md'}, page_content="# Shari Thomas – Professional and Academic Background\n\nShari Thomas is a technology professional with experience in higher education, software development, data science, and artificial intelligence. Her career reflects a combination of academic teaching, practical technical experience, and a continuing interest in developing skills relevant to the modern AI industry.\n\nShari completed her Bachelor of Technology (B.Tech) degree in Computer Science and Engineering. She subsequently obtained a Master of Technology (M.Tech) degree with a specialization in Image Processing. Her educational background introduced her to a range of computing disciplines, including software engineering, computer science fundamentals, digital image processing, and technologies associated with artificial intelligence.\n\nFollowing her postgraduate education, Shari began her professional career in the field of teaching. She wor

In [8]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 46 chunks
First chunk:

page_content='# Shari Thomas – Professional and Academic Background

Shari Thomas is a technology professional with experience in higher education, software development, data science, and artificial intelligence. Her career reflects a combination of academic teaching, practical technical experience, and a continuing interest in developing skills relevant to the modern AI industry.' metadata={'source': 'knowledge_base/Shari_Professional.md'}


PART B: Make vectors and store in Chroma

Huggingface embedding is one of the most popular one.

In [9]:
# Pick an embedding model

# Get Huggging face key.. this is not really required as we can directly call Huggingspace embedding 
HF_key = os.getenv('HF_API')
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16489.06it/s]


Vectorstore created with 46 documents


In [10]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 46 vectors with 384 dimensions in the vector store


In [13]:
# Visualization

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

# Pull everything out of Chroma (strings here, not Document objects)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
texts = result['documents']
metadatas = result['metadatas']

def get_category(source):
    parts = source.replace('\\', '/').split('/')   # handles Windows paths too
    folders = parts[1:-1]                          # drop 'knowledge_base' and filename
    return '/'.join(folders) if folders else 'overview (top level)'

df = pd.DataFrame({
    'source': [m['source'] for m in metadatas],
    'category': [get_category(m['source']) for m in metadatas],
    'file': [m['source'].split('/')[-1] for m in metadatas],
    'preview': [t[:100].replace('\n', ' ') + '...' for t in texts],
})

# t-SNE: perplexity MUST be lower than number of chunks
n = len(vectors)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, max(2, n // 4)))
df[['x', 'y']] = tsne.fit_transform(vectors)

fig = px.scatter(
    df, x='x', y='y',
    color='category',                     # automatic colors + legend
    hover_name='file',
    hover_data={'preview': True, 'x': False, 'y': False, 'category': False},
    title='2D Vector Store Visualization (t-SNE)',
    width=900, height=650,
)
fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.show()

In [14]:
import plotly.express as px
from sklearn.manifold import TSNE

# 3D t-SNE (same data, same perplexity logic as before)
n = len(vectors)
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, max(2, n // 4)))
coords_3d = tsne_3d.fit_transform(vectors)

df[['x3', 'y3', 'z3']] = coords_3d

fig = px.scatter_3d(
    df, x='x3', y='y3', z='z3',
    color='category',
    hover_name='file',
    hover_data={'preview': True, 'x3': False, 'y3': False, 'z3': False, 'category': False},
    title='3D Vector Store Visualization (t-SNE)',
    width=900, height=700,
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    margin=dict(r=10, b=10, l=10, t=40),
)
fig.show()